In [ ]:
%%bash
pip install numpy scipy matplotlib pandas torch scikit-learn --quiet

In [ ]:
import os
import glob
import struct
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import resample as scipy_resample
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
print('Imports OK')

In [ ]:
DATA_DIR       = r'C:/Users/quack/Documents/Projects/Verus/Data/Stephen Terracon Cornbread/Data'
WORKING_DIR    = r'C:/Users/quack/Documents/Projects/Verus/verus/server/models'
TARGET_SAMPLES = 512
MAX_DEPTH_MM   = 300.0  # TS picks are in mm
BATCH_SIZE     = 512
EPOCHS         = 100
PATIENCE       = 15
LR             = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
def preprocess_trace(raw_trace, target_samples=512):
    if len(raw_trace) != target_samples:
        raw_trace = scipy_resample(raw_trace, target_samples)
    raw_trace = raw_trace - raw_trace.mean()
    max_abs = np.abs(raw_trace).max()
    if max_abs > 0:
        raw_trace = raw_trace / max_abs
    return raw_trace.astype(np.float32)


def preprocess_batch(raw_traces, target_samples=512):
    if raw_traces.shape[1] != target_samples:
        raw_traces = scipy_resample(raw_traces, target_samples, axis=1)
    raw_traces = raw_traces - raw_traces.mean(axis=1, keepdims=True)
    norms = np.abs(raw_traces).max(axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return (raw_traces / norms).astype(np.float32)


def mm_to_normalized_depth(depth_mm, max_depth_mm=300.0):
    '''Normalize depth labels to 0-1. max_depth_mm=300 covers 0-30 cm.'''
    return np.clip(depth_mm / max_depth_mm, 0.0, 1.0)

In [ ]:
# Proceq RIS .scan binary format constants
# Layout: 'VH01SW' magic -> config name @ 0x000C -> D-blocks @ 0x027C
# Each D-block: 1036 bytes = 'D' + null + uint16(ch) + 8-byte ts + 510*int16 samples
# First 16 D-blocks are reference sweeps (zero payload).
_MAGIC        = b'VH01SW'
_D_START      = 0x027C
_D_SIZE       = 0x040C   # 1036 bytes per D-block
_D_HEADER     = 16
_D_SAMPLES    = (_D_SIZE - _D_HEADER) // 2  # 510 int16 samples
_D_REF_BLOCKS = 16
_D_MARKER     = b'D' + bytes(1)             # b'D\x00' start sentinel


def read_proceq_traces(scan_path):
    '''
    Read an odd-numbered Proceq RIS PRC .scan file.
    Returns (traces float32 shape (N, 510), n_traces).
    Returns (None, 0) for even files (no D-blocks) or unreadable files.
    '''
    with open(scan_path, 'rb') as f:
        raw = f.read()
    if raw[:6] != _MAGIC:
        return None, 0

    n_total = 0
    pos = _D_START
    while pos + 2 <= len(raw) and raw[pos:pos+2] == _D_MARKER:
        n_total += 1
        pos += _D_SIZE

    if n_total == 0:
        return None, 0   # even file, no D-blocks
    n_data = n_total - _D_REF_BLOCKS
    if n_data <= 0:
        return None, 0

    data_start = _D_START + _D_REF_BLOCKS * _D_SIZE
    traces = np.zeros((n_data, _D_SAMPLES), dtype=np.float32)
    with open(scan_path, 'rb') as f:
        f.seek(data_start)
        for i in range(n_data):
            block = f.read(_D_SIZE)
            if len(block) < _D_SIZE:
                traces = traces[:i]
                break
            s = np.frombuffer(block[_D_HEADER:], dtype='<i2').astype(np.float32)
            s -= s.mean()
            traces[i] = s

    norms = np.abs(traces).max(axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    traces /= norms
    return traces, len(traces)

In [ ]:
SKIP_SWATHS = {0, 4}  # swath 1: bottom-mat depths (229mm median); swath 5: corrupted TS (407-485mm)

def load_rebar_dataset():
    scan_files = sorted(glob.glob(os.path.join(DATA_DIR, "PRC_*.scan")))
    odd_scans  = [
        f for f in scan_files
        if int(os.path.basename(f).replace("PRC_", "").replace(".scan", "")) % 2 == 1
    ]
    ts_files = sorted(glob.glob(os.path.join(DATA_DIR, "TS_*_1.txt")))
    n_swaths = min(len(odd_scans) // 4, len(ts_files))
    good_swaths = [i for i in range(n_swaths) if i not in SKIP_SWATHS]
    print(f"Skipping swaths {sorted(s+1 for s in SKIP_SWATHS)} (bad/anomalous TS depths)")
    print(f"Using {len(good_swaths)} swaths: {[s+1 for s in good_swaths]}")

    all_traces, all_labels, all_swath_ids = [], [], []

    for sw_idx in good_swaths:
        swath_scans = odd_scans[sw_idx * 4 : sw_idx * 4 + 4]
        ts_picks    = np.loadtxt(ts_files[sw_idx])  # 3393 values in mm

        for scan_path in swath_scans:
            raw_traces, n_data = read_proceq_traces(scan_path)
            if raw_traces is None or n_data < 10:
                continue
            N         = len(raw_traces)
            x_ts      = np.linspace(0, N - 1, len(ts_picks))
            labels_mm = np.interp(np.arange(N), x_ts, ts_picks)
            processed = preprocess_batch(raw_traces, TARGET_SAMPLES)
            all_traces.append(processed)
            all_labels.append(mm_to_normalized_depth(labels_mm, MAX_DEPTH_MM).astype(np.float32))
            all_swath_ids.extend([sw_idx] * N)
            print(f"  sw{sw_idx+1:02d}  {os.path.basename(scan_path):25s}  "
                  f"{N:6d} traces  depth {labels_mm.mean():.1f} mm ({labels_mm.mean()/25.4:.2f} in)")

    return all_traces, all_labels, np.array(all_swath_ids), good_swaths


swath_traces, swath_labels, swath_ids, good_swaths = load_rebar_dataset()

all_traces = np.concatenate(swath_traces)
all_labels = np.concatenate(swath_labels)

# Val = last 3 good swaths; train = rest
val_sw_ids  = set(good_swaths[-3:])
train_mask  = np.array([sid not in val_sw_ids for sid in swath_ids])
val_mask    = ~train_mask

X_train, y_train = all_traces[train_mask], all_labels[train_mask]
X_val,   y_val   = all_traces[val_mask],   all_labels[val_mask]

depths_all_mm = all_labels * MAX_DEPTH_MM
print(f"
Dataset summary:")
print(f"  n_train_traces: {len(X_train):,}")
print(f"  n_val_traces:   {len(X_val):,}")
print(f"  depth range:    {depths_all_mm.min():.1f} - {depths_all_mm.max():.1f} mm")
print(f"  mean depth:     {(y_train * MAX_DEPTH_MM).mean():.1f} mm  "
      f"({(y_train * MAX_DEPTH_MM / 25.4).mean():.2f} in)")

In [ ]:
class GPRDataset(Dataset):
    def __init__(self, traces, labels, augment=False):
        self.traces  = torch.from_numpy(traces).unsqueeze(1)  # (N, 1, 512)
        self.labels  = torch.from_numpy(labels)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.traces[idx].clone()
        y = self.labels[idx]
        if self.augment:
            # Gaussian noise std=0.01
            if torch.rand(1) < 0.5:
                x = x + torch.randn_like(x) * 0.01
            # Amplitude scale 0.9-1.1
            if torch.rand(1) < 0.5:
                x = x * (0.9 + torch.rand(1) * 0.2)
            # Time shift +-10 samples with zero padding
            if torch.rand(1) < 0.5:
                shift = torch.randint(-10, 11, (1,)).item()
                x = torch.roll(x, shift, dims=-1)
                if shift > 0:
                    x[..., :shift] = 0.0
                elif shift < 0:
                    x[..., shift:] = 0.0
        return x, y


train_ds     = GPRDataset(X_train, y_train, augment=True)
val_ds       = GPRDataset(X_val,   y_val,   augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.score = nn.Linear(channels, 1)

    def forward(self, x):
        w = torch.softmax(self.score(x.transpose(1, 2)), dim=1)
        return (x.transpose(1, 2) * w).sum(dim=1)


class HorizonCNN(nn.Module):
    '''
    Rebar horizon depth regression.
    Input:  (batch, 1, 512) normalized trace
    Output: (batch,) normalized depth 0-1
    '''
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,   32,  7, padding=3), nn.ReLU(),
            nn.MaxPool1d(2),                               # 512->256
            nn.Conv1d(32,  64,  5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),                               # 256->128
            nn.Conv1d(64,  128, 3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),                               # 128->64
            nn.Conv1d(128, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),                               # 64->32
        )
        self.attn = TemporalAttention(128)
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()  # output 0-1
        )

    def forward(self, x):
        return self.head(self.attn(self.conv(x))).squeeze(-1)


model = HorizonCNN().to(DEVICE)
print(f'HorizonCNN parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

best_mae       = float('inf')
patience_count = 0
history        = []

print('  Ep   TR_loss  Val_loss  Val_MAE_cm  Val_RMSE_cm  Best_MAE          LR')
print('-' * 76)

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(train_ds)

    # Validate
    model.eval()
    val_loss, preds_v, tgts_v = 0.0, [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            val_loss += criterion(pred, yb).item() * len(yb)
            preds_v.append(pred.cpu().numpy())
            tgts_v.append(yb.cpu().numpy())
    val_loss /= len(val_ds)
    preds_v = np.concatenate(preds_v)
    tgts_v  = np.concatenate(tgts_v)

    mae_mm  = float(np.abs(preds_v - tgts_v).mean()           * MAX_DEPTH_MM)
    rmse_mm = float(np.sqrt(((preds_v - tgts_v)**2).mean())   * MAX_DEPTH_MM)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    if mae_mm < best_mae:
        best_mae, patience_count = mae_mm, 0
        torch.save(model.state_dict(), f'{WORKING_DIR}/horizon_model_best.pth')
    else:
        patience_count += 1

    history.append(dict(epoch=epoch, tr_loss=tr_loss, val_loss=val_loss,
                        mae_mm=mae_mm, rmse_mm=rmse_mm))

    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:4d}  {tr_loss:8.5f}  {val_loss:8.5f}  {mae_mm:10.3f}  '
              f'{rmse_mm:11.3f}  {best_mae:8.3f}  {lr:10.2e}')

    if patience_count >= PATIENCE:
        print(f'Early stop at epoch {epoch}')
        break

print(f'\nBest Val MAE: {best_mae:.3f} mm  ({best_mae / 25.4:.3f} in)')

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(f'{WORKING_DIR}/horizon_model_best.pth', map_location=DEVICE))
model.eval()

preds_v, tgts_v = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        preds_v.append(model(xb.to(DEVICE)).cpu().numpy())
        tgts_v.append(yb.numpy())

preds_mm = np.concatenate(preds_v) * MAX_DEPTH_MM
tgts_mm  = np.concatenate(tgts_v)  * MAX_DEPTH_MM

mae_mm  = float(np.abs(preds_mm - tgts_mm).mean())
rmse_mm = float(np.sqrt(((preds_mm - tgts_mm)**2).mean()))
mae_in  = mae_mm  / 2.54
rmse_in = rmse_mm / 2.54

# --- Scatter plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

lo, hi = tgts_mm.min(), tgts_mm.max()
axes[0].scatter(tgts_mm, preds_mm, alpha=0.02, s=1, rasterized=True)
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[0].set_xlabel('Actual depth (mm)')
axes[0].set_ylabel('Predicted depth (mm)')
axes[0].set_title(f'Predicted vs Actual\nMAE={mae_mm:.3f} cm  RMSE={rmse_mm:.3f} cm')

# --- Residual histogram ---
residuals = preds_mm - tgts_mm
axes[1].hist(residuals, bins=100, edgecolor='k', linewidth=0.3)
axes[1].axvline(0, color='r', lw=1.5)
axes[1].set_xlabel('Residual (mm)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residual distribution\nmean={residuals.mean():.3f}  std={residuals.std():.3f} cm')

# --- Loss curves ---
ep_h = [h['epoch'] for h in history]
axes[2].plot(ep_h, [h['tr_loss']  for h in history], label='Train')
axes[2].plot(ep_h, [h['val_loss'] for h in history], label='Val')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('SmoothL1 Loss')
axes[2].set_title('Training curves')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/rebar_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

# --- B-scan overlay: first 500 val traces ---
n_show = min(500, len(X_val))
x_show = torch.from_numpy(X_val[:n_show]).unsqueeze(1).to(DEVICE)
with torch.no_grad():
    pred_show = model(x_show).cpu().numpy() * MAX_DEPTH_MM
true_show = y_val[:n_show] * MAX_DEPTH_MM

ns_per_sample = 15.0 / TARGET_SAMPLES
velocity_m_ns = 0.15 / (6.0 ** 0.5)  # epsr=6.0 for IDS 2GHz on concrete

def mm_to_sample(mm_arr):
    return (mm_arr / 1000.0) / velocity_m_ns * 2.0 / ns_per_sample

fig, ax = plt.subplots(figsize=(16, 4))
ax.imshow(X_val[:n_show].T, aspect='auto', cmap='gray', vmin=-0.3, vmax=0.3, origin='upper')
x_idx = np.arange(n_show)
ax.plot(x_idx, mm_to_sample(true_show), 'r-', lw=0.8, label='TS ground truth')
ax.plot(x_idx, mm_to_sample(pred_show), 'b-', lw=0.8, label='Model prediction')
ax.set_xlabel('Trace index')
ax.set_ylabel('Sample index')
ax.set_title('B-scan overlay - first 500 val traces\nRed=TS picks, Blue=Model')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/bscan_overlay.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nFinal metrics:')
print(f'  MAE:  {mae_mm:.3f} cm  ({mae_in:.3f} in)')
print(f'  RMSE: {rmse_mm:.3f} cm  ({rmse_in:.3f} in)')

In [ ]:
import shutil
shutil.copy(f'{WORKING_DIR}/horizon_model_best.pth', f'{WORKING_DIR}/horizon_model.pth')
print(f'Saved: {WORKING_DIR}/horizon_model.pth')
print(f'MAE:  {mae_mm:.3f} cm  ({mae_in:.3f} in)')
print(f'RMSE: {rmse_mm:.3f} cm  ({rmse_in:.3f} in)')